<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_03_plate_transient_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 03 — The Plate with a Hole, in Time

**Paired with L8.2 · Dynamic Heat**

A plate with an elliptical hole, now transient. The plate starts cold, a uniform
source switches on at $t = 0$, the hole is held at zero, and the outer edges
are insulated, as in L8.1:

$$T_t = c\,(T_{xx} + T_{yy}) + Q$$

The late-time field should converge to the steady solution of the same plate —
**the steady state is the last frame of the transient**, and that is the claim
this notebook lets you check.

There is no exact solution here. From this notebook onward that is the normal
situation, and the checks have to change accordingly.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex08.2_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Points on a domain with a piece missing

`pb.plate_spacetime_points(..., with_hole=True)` samples the slab and rejects
anything inside the cylinder swept by the hole.

In [ ]:
C, Q, T_END = 1.0, 10.0, 1.0
N_F = 4000

set_seed(88)
model = MLP(n_in=3, n_hidden=40, n_layers=4)
describe(model, N_F)

xyt_f = to_tensor(pb.plate_spacetime_points(N_F, t_end=T_END, with_hole=True),
                  requires_grad=True)
xyt_o = to_tensor(boundary_points_in_time(25, 20, pb.PLATE_DOMAIN, (0.0, T_END),
                                          seed=1), requires_grad=True)
print("interior:", tuple(xyt_f.shape), "  outer edges:", tuple(xyt_o.shape))

## 2 · A trial solution combining the hole multiplier and the time factor

The level set vanishes on the hole and the factor $t$ vanishes at $t = 0$, so
one product enforces both:

$$T_{\mathrm{trial}} = t\,\phi(x,y)\,\mathcal{N}_{\hat w}(x,y,t)$$

The outer edges are insulated. That is a condition on a normal derivative,
which no multiplier can hold, so it stays a loss term: the zero-flux term of
Ex08.1 notebook 02, now sampled at every instant as well as along the edges.

### Your turn

In [ ]:
# TODO 1 --- the trial solution, hole and start built in ----------------------------------------------
# One `...` to replace:  t * pb.hole_multiplier(xy) * model(xyt)
#   t makes T = 0 at t = 0; the level set makes T = 0 on the hole. Slice xy out of xyt so the graph
#   runs back to xyt and the derivatives see both factors.
def trial(model, xyt):
    xy = xyt[:, 0:2]
    t = xyt[:, 2:3]
    return ...                                    # <- t * pb.hole_multiplier(xy) * model(xyt)
# ------------------------------------------------------------------------------

### Your turn, again — the residual and the loss

The source is inside the residual, not in a separate term. The only other term
is the zero flux through the insulated outer edges.

In [ ]:
# TODO 2 --- residual with a source, and the zero flux on the insulated edges ----------------------
# Three `...` to replace:
#   line 1  ->  grad(T, xyt)[:, 2:3] - C * (d2(T, xyt, 0) + d2(T, xyt, 1)) - Q      T_t = c (T_xx + T_yy) + Q
#   line 2  ->  (grad(T, xyt)[:, 0:2] ** 2).sum(dim=1, keepdim=True).mean()         x and y only: column 2 is d/dt
#   line 3  ->  mse(residual(model, xyt_f)) + W_FLUX * flux_loss(model, xyt_o)
# The zero-flux term is Ex08.1 notebook 02's, at every instant.
def residual(model, xyt):
    T = trial(model, xyt)
    return ...                                    # <- grad(T, xyt)[:, 2:3] - C * (d2(T, xyt, 0) + d2(T, xyt, 1)) - Q

W_FLUX = 1.0

def flux_loss(model, xyt):
    T = trial(model, xyt)
    return ...                                    # <- (grad(T, xyt)[:, 0:2] ** 2).sum(dim=1, keepdim=True).mean()

def loss_fn():
    return ...                                    # <- mse(residual(model, xyt_f)) + W_FLUX * flux_loss(model, xyt_o)
# ------------------------------------------------------------------------------

## 3 · Train

In [ ]:
history = train_two_stage(model, loss_fn, adam_steps=4000, lbfgs_steps=200,
                          lr=1e-3)
plot_curves(history, title="the plate with a hole, in time")
plt.show()

## 4 · The transient, frame by frame

In [ ]:
show = [0.02, 0.1, 0.5]
fig, axes = plt.subplots(1, len(show), figsize=(14, 3.6))
frames = {}
for a, t in zip(axes, show):
    X, Y, T = pb.slice_at_time(model, t, trial=trial, mask_hole=True)
    frames[t] = T
    pb.plot_slice(X, Y, T, ax=a, title=f"t = {t}", label="T")
plt.tight_layout(); plt.show()

for t in show:
    print(f"  t = {t:<5}  peak T = {np.nanmax(frames[t]):.4f}")

**Question.** Has the field reached its steady state by $t = 0.5$? Quote a
number, not an impression — the peak temperature and the position of the peak
will do, and in a steady state the heat leaving through the hole must equal the
heat generated in the plate.

Notebook 00's $\tau = 0.1013$ is for a slab held at zero on both faces. This
plate is insulated on its outer edges and loses heat only through the hole, so
it settles more slowly: a finite-difference estimate of its slowest mode gives
$\tau \approx 0.13$, which puts $t = 0.5$ at about four time constants, close to
steady but not certainly there. A finite-difference solution of the same
problem gives a peak of about 0.20, 0.84 and 1.58 at the three instants shown,
and 1.61 in the steady state. If your peaks fall short of those, the
interesting question is whether the physics or the network is responsible.

---

## 5 · Save

In [ ]:
os.makedirs(cc.OUTPUT_DIR, exist_ok=True)
path = os.path.join(cc.OUTPUT_DIR, "nb03_plate.npz")
np.savez(path, times=np.asarray(show),
         peak=np.asarray([np.nanmax(frames[t]) for t in show]),
         final=frames[show[-1]],
         adam=history["adam"], lbfgs=history["lbfgs"])
torch.save(model.state_dict(), os.path.join(cc.OUTPUT_DIR, "nb03_plate.pt"))
print("wrote", path)
cc.saved(path, os.path.join(cc.OUTPUT_DIR, "nb03_plate.pt"))


## 6 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Has the field reached its steady state by $t = 0.5$? Quote the peak temperatures you printed, not an impression. Section 4 puts this plate's time constant near 0.13, longer than both of notebook 00's: say why a plate that loses heat only through its hole settles more slowly than one held at zero on its edges, and what you would change if the peak were still rising.
   *→ L8.2 Q2*
2. The trial solution $t\,\phi\,\mathcal{N}$ has no $(1-t)f_{\mathrm{IC}}$ term. Say why it can drop it here, and what it still assumes about the range of $t$. Then list the conditions it enforces exactly and the one left to a loss term, and say why the insulated edges could not have been built into the trial solution the way the hole was.
   *→ L8.2 Q5*
3. There is no exact solution to score against. Name two checks you can still run. One should be the flux balance, `pb.flux_balance`, and the other a time history at the hottest point. Say when the flux balance applies in a transient, and what the history must do at $t = 0$ with this trial solution.
   *→ L8.2 Q6*
4. The source is constant. What would you change if $Q$ switched on and off with a duty cycle, and what would you expect the peak temperature to do? Say what a PINN offers here that a time-marching solver does not, if the on-time were made an extra input to the network.
   *→ L8.2 Q3*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 04](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_04_inverse_alpha_light.ipynb)**, where the diffusivity itself is the unknown.
